# Notebook 05 — Training a Tiny GPT-Style Model

**Goal:** Put the pieces together into a minimal language model, train it on a small corpus, and generate text.

We'll keep this notebook practical and short:

1. define the config
2. prepare the text data
3. train the model
4. generate a few samples

---

### Contents
1. [Configuration](#1)
2. [Prepare the corpus](#2)
3. [Train the model](#3)
4. [Generate text](#4)
5. [Key takeaways](#5)


In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import tiktoken
from tqdm.auto import tqdm
import sys

sys.path.insert(0, os.path.abspath('.'))
from utils.model import GPTModel

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps' if torch.backends.mps.is_available() else
    'cpu'
)

print('=' * 60)
print('  Notebook 05 — Training a Tiny GPT-Style Model')
print('=' * 60)
print(f'  Device : {device}')
print('=' * 60)


<a id='1'></a>
## 1 — Configuration

This config keeps the model small enough to train quickly, but all the main transformer pieces are still there.


In [ ]:
config = {
    'vocab_size': 50257,
    'd_model': 64,
    'n_heads': 4,
    'n_layers': 2,
    'd_ff': 256,
    'max_seq_len': 64,
    'dropout': 0.1,
    'batch_size': 16,
    'learning_rate': 3e-4,
    'num_steps': 300,
    'eval_interval': 50,
}

for k, v in config.items():
    print(f'{k:>14}: {v}')


<a id='2'></a>
## 2 — Prepare the corpus

We'll use a small engineering/science corpus. This is not enough for a high-quality model, but it is enough to demonstrate training and next-token prediction.


In [ ]:
corpus = """
Engineering is the application of scientific principles to design and build systems.
A control system measures the behaviour of a process and compares it with a desired target.
Embedded systems are designed to perform specific functions within larger machines.
Machine learning enables models to improve from data rather than explicit programming.
A transformer uses self attention to process sequences in parallel.
Functional safety is concerned with correct behaviour in response to inputs and faults.
Robotics combines sensors actuators and control to achieve useful physical behaviour.
Software engineering uses testing version control and automation to build reliable systems.
"""

enc = tiktoken.get_encoding('gpt2')
data = torch.tensor(enc.encode(corpus), dtype=torch.long)

split = int(0.9 * len(data))
train_data = data[:split]
val_data = data[split:]

print('Total tokens     :', len(data))
print('Training tokens  :', len(train_data))
print('Validation tokens:', len(val_data))


In [ ]:
def get_batch(data, batch_size, seq_len, device):
    starts = torch.randint(0, len(data) - seq_len - 1, (batch_size,))
    x = torch.stack([data[i:i+seq_len] for i in starts])
    y = torch.stack([data[i+1:i+seq_len+1] for i in starts])
    return x.to(device), y.to(device)

x, y = get_batch(train_data, batch_size=2, seq_len=8, device=device)
print('Input batch shape :', tuple(x.shape))
print('Target batch shape:', tuple(y.shape))


<a id='3'></a>
## 3 — Train the model


In [ ]:
model = GPTModel(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'])

train_losses = []
val_losses = []
val_steps = []

@torch.no_grad()
def estimate_loss(split_data, n_batches=10):
    model.eval()
    losses = []
    for _ in range(n_batches):
        xb, yb = get_batch(split_data, config['batch_size'], config['max_seq_len'], device)
        logits = model(xb)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), yb.view(-1))
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

start = time.time()
model.train()

for step in tqdm(range(config['num_steps'])):
    xb, yb = get_batch(train_data, config['batch_size'], config['max_seq_len'], device)
    logits = model(xb)
    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), yb.view(-1))

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    train_losses.append(loss.item())

    if step % config['eval_interval'] == 0:
        val_loss = estimate_loss(val_data, n_batches=5)
        val_losses.append(val_loss)
        val_steps.append(step)
        print(f'step {step:>3d} | train {loss.item():.4f} | val {val_loss:.4f}')

print(f'
Training finished in {time.time() - start:.1f}s')


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(train_losses, label='train', alpha=0.8)
ax.plot(val_steps, val_losses, 'o-', label='val')
ax.set_title('Training loss', fontweight='bold')
ax.set_xlabel('Step')
ax.set_ylabel('Cross-entropy loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
os.makedirs('checkpoints', exist_ok=True)
torch.save({'model_state_dict': model.state_dict(), 'config': config}, 'checkpoints/model.pt')
print('Saved checkpoint to checkpoints/model.pt')


<a id='4'></a>
## 4 — Generate text

Now we can sample from the model autoregressively.


In [ ]:
def generate(prompt, max_new_tokens=40, temperature=0.8, top_k=20):
    ids = torch.tensor([enc.encode(prompt)], dtype=torch.long, device=device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new_tokens,
                             temperature=temperature, top_k=top_k, greedy=False)
    return enc.decode(out[0].tolist())

for prompt in [
    'Engineering is',
    'A transformer',
    'Functional safety',
]:
    print('=' * 60)
    print('PROMPT:', repr(prompt))
    print(generate(prompt))
    print()


<a id='5'></a>
## 5 — Key takeaways

- We trained a small decoder-only transformer for **next-token prediction**
- The loss function is standard cross-entropy over the vocabulary
- The model is small, so the text quality is limited, but the full pipeline is real
- The saved checkpoint will be used in Notebook 06 for attention visualisation

If you want to scale it later, the main levers are:
- more data
- larger `d_model`
- more layers
- longer training
